# Student Performance Data Preprocessing

This notebook stops after the ingestion and preprocessing stage. It does not create visualizations or perform statistical analysis.

## 1. Ingest the raw CSV

Read `StudentsPerformance.csv` into a pandas DataFrame and inspect its structure.

In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd()
if BASE_DIR.name == 'src':
    BASE_DIR = BASE_DIR.parent
elif BASE_DIR.name != 'DV_Practice':
    BASE_DIR = BASE_DIR / 'DV_Practice'
DATA_PATH = BASE_DIR / 'data_raw' / 'StudentsPerformance.csv'
PROCESSED_PATH = BASE_DIR / 'data_clean' / 'processed_students_performance.csv'

raw_data = pd.read_csv(DATA_PATH)
print(f'Raw rows: {len(raw_data):,}')
print(f'Raw columns: {list(raw_data.columns)}')
display(raw_data.head())

Raw rows: 1,000
Raw columns: ['gender', 'race/ethnicity', 'parental level of education', 'lunch', 'test preparation course', 'math score', 'reading score', 'writing score']


,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


## 2. Preprocess the data

Standardize column names, clean categorical values, convert scores to numeric values, handle missing values, and calculate `overall_avg` from the three subject scores.

In [2]:
data = raw_data.rename(columns={
    'race/ethnicity': 'race_ethnicity',
    'parental level of education': 'parental_education',
    'test preparation course': 'test_prep',
    'math score': 'math_score',
    'reading score': 'reading_score',
    'writing score': 'writing_score',
}).copy()

text_columns = ['gender', 'race_ethnicity', 'parental_education', 'lunch', 'test_prep']
score_columns = ['math_score', 'reading_score', 'writing_score']

for column in text_columns:
    data[column] = data[column].astype('string').str.strip()
    data[column] = data[column].replace({'': pd.NA, 'nan': pd.NA})

for column in score_columns:
    data[column] = pd.to_numeric(data[column], errors='coerce')

missing_before = int(data.isna().sum().sum())
required_columns = text_columns + score_columns
data = data.dropna(subset=required_columns).copy()
data['overall_avg'] = data[score_columns].mean(axis=1).round(2)

data['gender'] = pd.Categorical(data['gender'], categories=['female', 'male'])
data['lunch'] = pd.Categorical(data['lunch'], categories=['standard', 'free/reduced'])
data['test_prep'] = pd.Categorical(data['test_prep'], categories=['none', 'completed'])

rows_removed = len(raw_data) - len(data)
data.to_csv(PROCESSED_PATH, index=False)

print(f'Missing values before cleaning: {missing_before}')
print(f'Rows removed: {rows_removed}')
print(f'Processed rows: {len(data):,}')
print(f'Saved: {PROCESSED_PATH.name}')
display(data.head())
display(data.dtypes)

Missing values before cleaning: 0
Rows removed: 0
Processed rows: 1,000
Saved: processed_students_performance.csv


,gender,race_ethnicity,parental_education,lunch,test_prep,math_score,reading_score,writing_score,overall_avg
0,female,group B,bachelor's degree,standard,none,72,72,74,72.67
1,female,group C,some college,standard,completed,69,90,88,82.33
2,female,group B,master's degree,standard,none,90,95,93,92.67
3,male,group A,associate's degree,free/reduced,none,47,57,44,49.33
4,male,group C,some college,standard,none,76,78,75,76.33


gender                      category
race_ethnicity        string[python]
parental_education    string[python]
lunch                       category
test_prep                   category
math_score                     int64
reading_score                  int64
writing_score                  int64
overall_avg                  float64
dtype: object